# 05 - Object-Oriented Python

**Sources:** Bootcamp BP4 (*A Pythonic Object, Data Classes, Interfaces/Protocols/ABCs, Inheritance,
Operator Overloading, Context Managers* - headings only in the PDF) and Summer School
*CursP_BasePython* (class declaration, class extension) + *CursP_sklearn* (the `Diputado` /
`Parlament` classes of the eigenfaces session).

**Contents:** classes & instances -> special methods ("dunders") -> properties -> class/static methods ->
dataclasses -> inheritance, `super()`, MRO -> ABCs & Protocols -> operator overloading ->
context managers -> composition vs inheritance.

## 1. A first class (the slide example)

In [1]:
class Student(object):          # `(object)` is implicit in Python 3
    def __init__(self, name):
        self.name = name

    def set_age(self, age):
        self.age = age

    def set_major(self, major):
        self.major = major

anna = Student("anna")
anna.set_age(21)
anna.set_major("physics")
print(anna.name, anna.age, anna.major)
print(anna.__dict__)             # instance attributes live in a dict

anna 21 physics
{'name': 'anna', 'age': 21, 'major': 'physics'}


In [2]:
class MasterStudent(Student):
    internship = "mandatory, from March to June"       # class attribute (shared by all instances)

james = MasterStudent("james")
print(james.internship, MasterStudent.internship)
james.set_age(23)
print(james.age, isinstance(james, Student), issubclass(MasterStudent, Student))

mandatory, from March to June mandatory, from March to June
23 True True


## 2. A "Pythonic object"
Python objects cooperate with the language through **special methods** (`__dunder__`):

| Method | Used by |
|---|---|
| `__repr__` | developers: `repr(x)`, REPL display, debugging. Aim for unambiguous / `eval`-able |
| `__str__` | users: `str(x)`, `print(x)`, f-strings (falls back to `__repr__`) |
| `__eq__`, `__hash__` | `==`, use as dict key / set member |
| `__lt__` ... | ordering, `sorted` |
| `__len__`, `__getitem__`, `__iter__`, `__contains__` | container behaviour |
| `__bool__` | truthiness |
| `__call__` | calling the object like a function |
| `__enter__`, `__exit__` | `with` statement |

In [3]:
class Vector2D:
    """A 2-D vector - a small but complete 'Pythonic' value object."""

    __slots__ = ("x", "y")                  # no per-instance __dict__: less memory, no typos

    def __init__(self, x=0.0, y=0.0):
        self.x = float(x)
        self.y = float(y)

    def __repr__(self):
        return f"Vector2D({self.x!r}, {self.y!r})"

    def __str__(self):
        return f"({self.x:g}, {self.y:g})"

    def __eq__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented            # let Python try the reflected operation / fall back
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))        # equal objects MUST have equal hashes

    def __bool__(self):
        return bool(self.x or self.y)

    def __iter__(self):                      # enables unpacking: x, y = v
        yield self.x
        yield self.y

    # --- operator overloading ---
    def __add__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector2D(self.x * scalar, self.y * scalar)

    __rmul__ = __mul__                       # makes `2 * v` work too

    def __neg__(self):
        return Vector2D(-self.x, -self.y)

    def __abs__(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

    def __matmul__(self, other):             # `@` as the dot product
        return self.x * other.x + self.y * other.y

v, w = Vector2D(3, 4), Vector2D(1, 2)
print(repr(v), str(v), v + w, v - w, v * 2, 2 * v, -v, abs(v), v @ w)
x, y = v
print(x, y, v == Vector2D(3, 4), v == (3, 4), {v: "a dict key"}, bool(Vector2D()))

Vector2D(3.0, 4.0) (3, 4) (4, 6) (2, 2) (6, 8) (6, 8) (-3, -4) 5.0 11.0
3.0 4.0 True False {Vector2D(3.0, 4.0): 'a dict key'} False


Returning `NotImplemented` (not raising) is the convention: Python then tries the other operand's reflected
method (`__radd__`, `__rmul__`, ...) before raising `TypeError`.

In [4]:
try:
    v + 5
except TypeError as err:
    print("TypeError:", err)

TypeError: unsupported operand type(s) for +: 'Vector2D' and 'int'


## 3. Attributes, properties, class & static methods
* **Instance attributes** - per object. **Class attributes** - shared; looked up if the instance lacks one.
* Leading underscore `_x` = "internal, please don't touch" (convention). Double `__x` triggers name mangling.
* `@property` - computed / validated attribute with plain `obj.attr` syntax (don't write Java-style getters).
* `@classmethod` - receives the class (alternative constructors). `@staticmethod` - plain function in the class namespace.

In [5]:
class Temperature:
    scale = "Celsius"                          # class attribute
    instances = 0

    def __init__(self, celsius):
        self.celsius = celsius                 # goes through the property setter
        Temperature.instances += 1

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

    @property
    def fahrenheit(self):                      # read-only computed property
        return self._celsius * 9 / 5 + 32

    @classmethod
    def from_fahrenheit(cls, f):               # alternative constructor
        return cls((f - 32) * 5 / 9)

    @staticmethod
    def is_valid(value):
        return value >= -273.15

t = Temperature(25)
print(t.celsius, t.fahrenheit, Temperature.from_fahrenheit(212).celsius, Temperature.is_valid(-300))
try:
    t.celsius = -500
except ValueError as err:
    print("ValueError:", err)
try:
    t.fahrenheit = 3
except AttributeError as err:
    print("AttributeError:", err)
print(Temperature.instances)

25.0 77.0 100.0 False
ValueError: below absolute zero
AttributeError: property 'fahrenheit' of 'Temperature' object has no setter
2


## 4. Data classes
`@dataclass` generates `__init__`, `__repr__`, `__eq__` (and optionally ordering, hashing, immutability)
from annotated fields - perfect for "bags of data".

In [6]:
from dataclasses import dataclass, field, asdict, astuple, replace, fields

@dataclass(order=True)                          # adds <, <=, >, >= comparing fields in order
class Diputado:
    """A member of parliament (the class of the eigenfaces session, modernised)."""
    ind: int
    name: str = ""
    surname: str = ""
    gender: str = ""
    party: str = ""
    tags: list[str] = field(default_factory=list, compare=False)   # NEVER use a bare mutable default

    def __post_init__(self):                    # validation / derived fields after __init__
        self.name = self.name.strip()
        self.surname = self.surname.strip()

    @property
    def full_name(self) -> str:
        return f"{self.name} {self.surname}".strip()

d1 = Diputado(1, " Ana ", "Lopez", "M", "GS")
d2 = Diputado(2, "Bob", "Smith", "H", "GP", tags=["new"])
print(d1)
print(d1 == Diputado(1, "Ana", "Lopez", "M", "GS"), d1 < d2, d1.full_name)
print(asdict(d2), astuple(d1)[:3], replace(d1, party="GP").party, [f.name for f in fields(Diputado)])

Diputado(ind=1, name='Ana', surname='Lopez', gender='M', party='GS', tags=[])
True True Ana Lopez
{'ind': 2, 'name': 'Bob', 'surname': 'Smith', 'gender': 'H', 'party': 'GP', 'tags': ['new']} (1, 'Ana', 'Lopez') GP ['ind', 'name', 'surname', 'gender', 'party', 'tags']


In [7]:
@dataclass(frozen=True, slots=True)              # immutable + hashable + memory-light
class Point:
    x: float
    y: float

p = Point(1, 2)
try:
    p.x = 5
except Exception as err:
    print(type(err).__name__, "-", err)
print({p: "usable as a key"}, p)

FrozenInstanceError - cannot assign to field 'x'
{Point(x=1, y=2): 'usable as a key'} Point(x=1, y=2)


In [8]:
# A container class that behaves like a sequence (the `Parlament` class of the slides)
class Parlament:
    def __init__(self, members=()):
        self._members: list[Diputado] = list(members)

    def add(self, member: Diputado) -> None:
        self._members.append(member)

    def __len__(self):
        return len(self._members)

    def __getitem__(self, key):                  # supports p[0], p[-1] AND slices p[1:3]
        return self._members[key]

    def __iter__(self):
        return iter(self._members)

    def __contains__(self, item):
        return item in self._members

    def __repr__(self):
        return f"Parlament({len(self)} members)"

    def by_gender(self, gender):
        return [m for m in self if m.gender == gender]

parl = Parlament([d1, d2])
parl.add(Diputado(3, "Cy", "Doe", "H", "GS"))
print(parl, len(parl), parl[0].full_name, [m.name for m in parl[1:]], d2 in parl, len(parl.by_gender("H")))

Parlament(3 members) 3 Ana Lopez ['Bob', 'Cy'] True 2


## 5. Inheritance, `super()`, MRO, duck typing
* `class Child(Parent)`; call the parent with `super()`.
* **MRO** (method resolution order, C3 linearisation) decides which method wins with multiple inheritance:
  `Class.__mro__`.
* **Duck typing**: "if it quacks like a duck" - code relies on behaviour, not declared type.
* Rule of thumb: **prefer composition over inheritance** ("has-a" before "is-a").

In [9]:
class Animal:
    sound = "..."

    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} says {self.sound}"

class Dog(Animal):
    sound = "woof"

    def __init__(self, name, tricks=()):
        super().__init__(name)                   # reuse the parent's initialiser
        self.tricks = list(tricks)

    def speak(self):                             # override + extend
        return super().speak() + "!"

class Cat(Animal):
    sound = "meow"

for pet in [Dog("Rex", ["sit"]), Cat("Tom"), Animal("Thing")]:
    print(pet.speak())

print(Dog.__mro__)

Rex says woof!
Tom says meow
Thing says ...
(<class '__main__.Dog'>, <class '__main__.Animal'>, <class 'object'>)


In [10]:
# Multiple inheritance & the diamond: cooperative super() calls follow the MRO
class A:
    def hello(self):
        return ["A"]

class B(A):
    def hello(self):
        return ["B"] + super().hello()

class C(A):
    def hello(self):
        return ["C"] + super().hello()

class D(B, C):
    def hello(self):
        return ["D"] + super().hello()

print(D().hello(), [k.__name__ for k in D.__mro__])

['D', 'B', 'C', 'A'] ['D', 'B', 'C', 'A', 'object']


In [11]:
# Mixins: small, focused classes that add one capability
import json

class JsonMixin:
    def to_json(self):
        return json.dumps(self.__dict__)

class Config(JsonMixin):
    def __init__(self, **kw):
        self.__dict__.update(kw)

print(Config(debug=True, retries=3).to_json())

{"debug": true, "retries": 3}


## 6. Interfaces: ABCs and Protocols

* **Abstract Base Class** (`abc.ABC`): nominal interface. Subclasses *must* implement `@abstractmethod`s,
  or they cannot be instantiated.
* **Protocol** (`typing.Protocol`): structural interface - any class with the right methods matches,
  no inheritance required (checked by `mypy`; add `@runtime_checkable` for `isinstance`).

In [12]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

    @abstractmethod
    def perimeter(self) -> float: ...

    def describe(self) -> str:                    # concrete method shared by all shapes ("template")
        return f"{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2
    def perimeter(self): return 2 * math.pi * self.r

class Rect(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h
    def perimeter(self): return 2 * (self.w + self.h)

try:
    Shape()
except TypeError as err:
    print("TypeError:", err)

for s in [Circle(1), Rect(2, 3)]:
    print(s.describe())

class Broken(Shape):
    def area(self): return 0              # forgot perimeter()
try:
    Broken()
except TypeError as err:
    print("TypeError:", err)

TypeError: Can't instantiate abstract class Shape without an implementation for abstract methods 'area', 'perimeter'
Circle: area=3.14, perimeter=6.28
Rect: area=6.00, perimeter=10.00
TypeError: Can't instantiate abstract class Broken without an implementation for abstract method 'perimeter'


In [13]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> str: ...

class Sprite:                              # does NOT inherit from Drawable
    def draw(self) -> str:
        return "sprite!"

def render(item: Drawable) -> str:
    return item.draw()

print(render(Sprite()), isinstance(Sprite(), Drawable), isinstance(42, Drawable))

# Built-in ABCs describe common interfaces
from collections.abc import Iterable, Sequence, Mapping, MutableMapping
print(isinstance([], Sequence), isinstance({}, Mapping), isinstance("s", Iterable))

sprite! True False
True True True


## 7. Operator overloading & rich comparisons
Beyond arithmetic: `__lt__ + @total_ordering`, `__getitem__`, `__call__`, `__contains__`, `__len__`.

In [14]:
from functools import total_ordering

@total_ordering                                 # define __eq__ and ONE ordering method; the rest are derived
class Version:
    def __init__(self, text):
        self.parts = tuple(int(p) for p in text.split("."))
        self.text = text

    def __eq__(self, other):
        return self.parts == other.parts

    def __lt__(self, other):
        return self.parts < other.parts

    def __repr__(self):
        return f"Version({self.text!r})"

versions = [Version("1.10.0"), Version("1.2.9"), Version("1.2.10")]
print(sorted(versions), max(versions), Version("1.0") <= Version("1.0"))

class Polynomial:
    def __init__(self, *coeffs):               # coeffs from highest degree: 3x^2 + 2x - 1 -> (3, 2, -1)
        self.coeffs = coeffs

    def __call__(self, x):                      # callable object, evaluated with Horner's rule
        result = 0
        for c in self.coeffs:
            result = result * x + c
        return result

p = Polynomial(3, 2, -1)
print(p(0), p(2), callable(p))

[Version('1.2.9'), Version('1.2.10'), Version('1.10.0')] Version('1.10.0') True
-1 15 True


## 8. Context managers
The `with` statement guarantees set-up/tear-down. Protocol: `__enter__()` (value bound by `as`) and
`__exit__(exc_type, exc, tb)` (return `True` to *swallow* the exception).

In [15]:
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"block took {self.elapsed * 1000:.2f} ms")
        return False                             # don't suppress exceptions

with Timer() as timer:
    sum(i * i for i in range(100_000))
print(timer.elapsed > 0)

class Suppress:
    def __init__(self, *exc_types):
        self.exc_types = exc_types

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        return exc_type is not None and issubclass(exc_type, self.exc_types)

with Suppress(ZeroDivisionError):
    1 / 0
print("still running")

block took 5.08 ms
True
still running


In [16]:
# The easy way: contextlib
from contextlib import contextmanager, suppress, redirect_stdout, closing, ExitStack
import io, os, tempfile

@contextmanager
def working_directory(path):
    previous = os.getcwd()
    os.chdir(path)
    try:
        yield path                               # the `with` body runs here
    finally:
        os.chdir(previous)                       # always restored, even on error

with working_directory(tempfile.gettempdir()) as where:
    print("inside:", os.path.basename(os.getcwd()) == os.path.basename(where))
print("restored:", os.getcwd() != where)

buffer = io.StringIO()
with redirect_stdout(buffer):
    print("captured!")
print(repr(buffer.getvalue()))

with suppress(FileNotFoundError):
    os.remove("definitely-not-here.txt")

inside: True
restored: True
'captured!\n'


## 9. Composition vs inheritance - quick design guide

* *"is-a"* and Liskov-substitutable? -> inheritance may be fine (`Dog` is an `Animal`).
* *"has-a"* / *"uses-a"*? -> **composition**: hold a collaborator and delegate.
* Need only shared behaviour across unrelated classes? -> mixin, protocol, or plain functions.
* Python modules and functions are objects too: **you do not need a class for everything**.

In [17]:
class Engine:
    def start(self): return "vroom"

class Car:
    def __init__(self, engine=None):
        self.engine = engine or Engine()          # dependency injection -> easy to test with a fake engine
    def start(self):
        return f"car says {self.engine.start()}"

class SilentEngine:
    def start(self): return "..."

print(Car().start(), Car(SilentEngine()).start())

car says vroom car says ...


## Exercises

**A. Bank account.** `Account` with `balance` property (never negative), `deposit`, `withdraw`
(raise `InsufficientFunds` - your own exception), `__repr__`, and ordering by balance. Add a `SavingsAccount`
subclass with an `add_interest()` method.

**B. Fraction class.** Implement `Fraction(num, den)` with normalisation (gcd), `+ - * /`, comparison,
`__float__`, `__repr__`. Test against `fractions.Fraction`.

**C. Plugin registry.** Define an ABC `Exporter` with `export(data) -> str`; write `CsvExporter` and `JsonExporter`;
keep a registry dict filled by a class decorator.

**D. Matrix mini-class.** `Matrix` storing a list of lists with `__getitem__((i, j))`, `__matmul__`,
`transpose()`, `__iter__`, and compare results against NumPy.

**E. Rewrite `Diputado`/`Parlament`** so that `Parlament.from_csv(path)` is a `@classmethod` and iteration is
lazy (generator). This feeds project 04.

In [18]:
# Your turn